# Figure 4 — Stability and ML forcing

Global pressure, moisture, energy, and ML-tendency behavior.


## 1. Load only the required common-grid data


In [ ]:
from pathlib import Path
import sys
import numpy as np
import xarray as xr
from dask.distributed import Client, get_client

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run from ml_implement_paper/ or its notebooks/ directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src import io as data_io
from src import metrics, plotting, preprocessing

config = data_io.load_config(PROJECT_ROOT / 'config' / 'paths.yaml')
paths = data_io.output_paths(config)
analysis = config['analysis']

# Reuse an existing client (for example, one supplied by Jupyter) or start a
# conservative local threaded client. Xarray operations remain lazy until save.
try:
    client = get_client()
except ValueError:
    client = Client(
        n_workers=4,
        threads_per_worker=1,
        processes=False,
        dashboard_address=':0',
    )
client

ML_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/ml_method_2026')
REFERENCE_ROOT = Path('/pscratch/sd/z/zhan391/seacrogs_scratch/reference_nudge')
POST_SUBDIR = Path('post/atm/180x360_aave/ts/3hourly/1yr')
PERIOD = '201201_201212'

CASE_DIRS = {
    'CTRL': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_CTRL',
    'UNET': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNET_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'UNETXTR': ML_ROOT / 'F20TR_NE30PG2_EC30TO60E2R2_UVTQ_UNETXTR_IMT_NS6_WOQMADJ_WOTVCON_PBL222_WOVSMOOTH',
    'REF': REFERENCE_ROOT / 'F20TR_ne30pg2_EC30to60E2r2_NDGUVTQ_IMT_3hr_pm-cpu_08-01-25',
}
CASE_DIRS = {name: CASE_DIRS[name] for name in ['CTRL', 'UNET', 'UNETXTR']}
REQUIRED_VARIABLES = ['PS', 'TMQ', 'FSNT', 'FLNT', 'Nudge_U', 'Nudge_V', 'Nudge_T', 'Nudge_Q']

STATE_VARIABLES = ['PS', 'TMQ', 'FSNT', 'FLNT']
TENDENCY_VARIABLES = ['Nudge_U', 'Nudge_V', 'Nudge_T', 'Nudge_Q']
REQUIRED_BY_CASE = {
    name: STATE_VARIABLES + (TENDENCY_VARIABLES if name != 'CTRL' else [])
    for name in CASE_DIRS
}
files = {
    name: [case_dir / POST_SUBDIR / f'{variable}_{PERIOD}.nc' for variable in REQUIRED_BY_CASE[name]]
    for name, case_dir in CASE_DIRS.items()
}
missing = {name: [str(path) for path in paths_ if not path.exists()] for name, paths_ in files.items()}
missing = {name: paths_ for name, paths_ in missing.items() if paths_}
if missing:
    details = '\n'.join(f'  {name}: {len(paths_)} missing file(s)' for name, paths_ in missing.items())
    raise FileNotFoundError('Postprocess the required variables first:\n' + details)

chunks = {'time': 32, 'lat': 45, 'lon': 90}
datasets = {
    name: xr.merge([xr.open_dataset(path, chunks=chunks, cache=False) for path in paths_], join='exact')
    for name, paths_ in files.items()
}
datasets = {
    name: preprocessing.subset_time(ds, analysis['start_date'], analysis['end_date'])
    for name, ds in datasets.items()
}
datasets = preprocessing.match_common_times(datasets)
datasets = {name: preprocessing.daily_mean(ds) for name, ds in datasets.items()}

sample = datasets[next(iter(datasets))][REQUIRED_VARIABLES[0]].isel(time=0, drop=True)
area = np.cos(np.deg2rad(sample['lat'])).clip(min=0).broadcast_like(sample)
area = area / area.sum()
{name: dict(ds.sizes) for name, ds in datasets.items()}

## 2. Quality control


In [ ]:
# Run all finite-value checks together so Dask can share I/O efficiently.
import dask

qc_keys = []
qc_tasks = []
for case_name, dataset in datasets.items():
    for variable in dataset.data_vars:
        qc_keys.append((case_name, variable))
        qc_tasks.extend([
            np.isfinite(dataset[variable]).any().data,
            (~np.isfinite(dataset[variable])).sum().data,
        ])
qc_values = dask.compute(*qc_tasks)
qc = {}
for index, key in enumerate(qc_keys):
    has_finite = bool(qc_values[2 * index])
    invalid_count = int(qc_values[2 * index + 1])
    if not has_finite:
        raise ValueError(f'{key[0]}:{key[1]} contains no finite values')
    qc.setdefault(key[0], {})[key[1]] = invalid_count
qc

## 3. Process and save the diagnostic data


In [ ]:
experiments = {name: datasets[name] for name in ('CTRL', 'UNET', 'UNETXTR')}
diagnostic = metrics.stability_diagnostic(
    experiments, area, tendencies=['Nudge_U', 'Nudge_V', 'Nudge_T', 'Nudge_Q']
)
data_io.save_dataset(diagnostic, paths['processed'] / 'stability_timeseries.nc')
diagnostic

## 4. Reload the diagnostic product and create the figure


In [ ]:
diagnostic = xr.open_dataset(paths['processed'] / 'stability_timeseries.nc')
plotting.plot_stability(diagnostic, paths['figures'] / 'fig04_stability.png')